# 实验4：AlexNet的网络结构及其实现代码

本实验基于教材第5章代码示例 5-5 ~ 5-8，学习AlexNet的网络结构并使用TensorFlow/Keras实现。

## 一、实验目的

1. 了解AlexNet的历史意义和在深度学习发展中的关键地位
2. 掌握AlexNet的网络结构及各层设计思想
3. 使用TensorFlow/Keras实现AlexNet
4. 理解Dropout正则化的作用

## 二、AlexNet简介

### ImageNet竞赛与AlexNet的历史性突破

ImageNet大规模视觉识别挑战赛（ILSVRC）是计算机视觉领域最权威的竞赛之一。该数据集包含约**120万张**训练图像、5万张验证图像和15万张测试图像，涵盖**1000个类别**（如不同品种的狗、各类交通工具、日常物品等）。

在AlexNet出现之前，参赛者主要使用传统方法（如SIFT特征+SVM分类器），Top-5错误率在**26%**左右徘徊。2012年，Alex Krizhevsky、Ilya Sutskever和Geoffrey Hinton提出的AlexNet将Top-5错误率骤降至**15.3%**，领先第二名超过10个百分点。这一结果震惊了整个学术界，标志着深度学习时代的正式开启。

### 关键创新

- 使用 **ReLU** 激活函数（替代Sigmoid/Tanh），大幅加速训练
- 使用 **Dropout** 防止过拟合
- 使用 **数据增强** 扩充训练数据
- 利用 **GPU** 加速训练（2块NVIDIA GTX 580并行训练）

### ReLU激活函数

AlexNet首次在大规模网络中使用ReLU激活函数，其公式为：

$$f(x) = \max(0, x)$$

**ReLU相比Sigmoid/Tanh的优势：**
1. **无梯度消失**：在正区间梯度恒为1，保证梯度能有效传播到深层网络。Sigmoid最大梯度仅为0.25，多层连乘后梯度指数衰减。
2. **计算高效**：仅需与0比较，无需指数运算。
3. **稀疏激活**：部分神经元输出为0，产生稀疏表示，有助于特征解耦。
4. **收敛更快**：实验表明ReLU网络达到相同错误率的速度比Tanh快约**6倍**。

ReLU函数图形非常直观：在x轴负半轴输出为0（一条水平线），在正半轴为y=x（一条45度斜线），在原点处有一个拐点。

### 网络结构总览

| 层号 | 层类型 | 输出形状 | 说明 |
|------|--------|----------|------|
| 输入 | Input | 227x227x3 | RGB彩色图像 |
| C1 | Conv2D(96, 11x11, stride=4) + ReLU | 55x55x96 | 大卷积核提取低级特征 |
| S1 | MaxPooling(3x3, stride=2) | 27x27x96 | 下采样 |
| C2 | Conv2D(256, 5x5, same) + ReLU | 27x27x256 | 提取中级特征 |
| S2 | MaxPooling(3x3, stride=2) | 13x13x256 | 下采样 |
| C3 | Conv2D(384, 3x3, same) + ReLU | 13x13x384 | 提取高级特征 |
| C4 | Conv2D(384, 3x3, same) + ReLU | 13x13x384 | 进一步提取特征 |
| C5 | Conv2D(256, 3x3, same) + ReLU | 13x13x256 | 提取高级特征 |
| S5 | MaxPooling(3x3, stride=2) | 6x6x256 | 下采样 |
| F6 | Flatten + Dense(4096) + ReLU + Dropout(0.5) | 4096 | 全连接 |
| F7 | Dense(4096) + ReLU + Dropout(0.5) | 4096 | 全连接 |
| 输出 | Dense(N) + Softmax | N | N类分类 |

## 三、Dropout 正则化

### 过拟合问题

当模型参数量远大于训练数据量时，模型容易"记住"训练数据中的噪声和细节，而非学习通用规律。过拟合的典型表现是：**训练损失很低但测试损失很高**，训练准确率接近100%但测试准确率明显较低。AlexNet有约6000万个参数，极易过拟合，因此需要有效的正则化手段。

### Dropout的工作原理

**训练阶段：** 在每次前向传播时，以概率 $p$ 随机将神经元的输出置为0（"丢弃"该神经元）。被丢弃的神经元不参与本次的前向传播和反向传播。每次训练步骤实际上使用的是原始网络的一个"子网络"，不同步骤使用不同的子网络结构。

**测试阶段：** 不进行任何丢弃，所有神经元都参与计算。Keras使用"inverted dropout"策略：在训练时将保留神经元的输出除以 $(1-p)$ 进行缩放，因此测试时不需要额外缩放。

### 为什么 p=0.5 最常用？

当 $p=0.5$ 时，可能产生的子网络数量最多（$2^n$ 种组合，其中 $n$ 为神经元数），正则化效果最强。AlexNet在两个全连接层（F6、F7）后各使用了 Dropout(0.5)，即训练时随机丢弃50%的神经元。

### 集成学习的解释

Dropout可以理解为一种隐式的**模型集成（Ensemble）**。训练过程中，每个mini-batch实际上是在训练一个不同的子网络。最终测试时使用完整网络，相当于对指数级数量的子网络的预测结果取平均，从而获得更好的泛化能力。这类似于随机森林中多棵决策树投票的思想。

通过随机丢弃神经元，Dropout迫使每个神经元不能依赖特定的其他神经元（防止"共适应"），从而学习到更加鲁棒、独立的特征表示。

---
## 四、实验步骤

### 步骤1：导入依赖

导入实验所需的库：NumPy用于数值计算，Matplotlib用于可视化，TensorFlow/Keras用于构建和训练神经网络。

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

# [旧写法] from keras.layers import Activation, Conv2D, BatchNormalization, Dense
# [旧写法] from keras.layers import Dropout, Flatten, Input, MaxPooling2D, ZeroPadding2D
# [旧写法] from keras import Model
# ↑ 独立 keras 包在 TF 2.0+ 中已不推荐使用

# [新写法] 适用于 TensorFlow >= 2.0
import tensorflow as tf
from tensorflow.keras.layers import Activation, Conv2D, BatchNormalization, Dense
from tensorflow.keras.layers import Dropout, Flatten, Input, MaxPooling2D, ZeroPadding2D
from tensorflow.keras import Model

print(f'TensorFlow version: {tf.__version__}')

### 步骤2：数据准备（参考代码示例 5-5）

原书使用 `ImageDataGenerator.flow_from_directory()` 从本地文件夹加载图像数据。  
由于实验环境中可能没有对应的数据集目录，这里展示原始写法供参考，**实际不运行**。

**关于 `ImageDataGenerator` 的说明：**

- **惰性加载（Lazy Loading）**：`ImageDataGenerator` 不会一次性将所有图像加载到内存中，而是在训练时按需从磁盘读取每个batch的数据。这对于大型数据集（如ImageNet的120万张图像）至关重要，因为它们无法全部放入内存。
- **内存高效**：每次只将一个batch的图像（如200张）加载到内存中，训练完成后即释放，显著降低内存占用。
- **`rescale=1./255`**：将像素值从 [0, 255] 归一化到 [0, 1]，有助于训练收敛和数值稳定性。
- **`flow_from_directory()`**：从文件夹结构自动推断类别标签，每个子文件夹名即为类别名（如 `data/train/cat/`、`data/train/dog/`）。
- **`target_size=(227, 227)`**：自动将所有图像缩放到AlexNet要求的输入尺寸。
- **`class_mode='categorical'`**：生成one-hot编码的标签，适配 `categorical_crossentropy` 损失函数。

In [ ]:
# ====== 代码示例 5-5（展示用，需要本地数据目录才能运行） ======

# [旧写法] from keras.preprocessing.image import ImageDataGenerator
# ↑ 独立 keras 包在 TF 2.0+ 中已不推荐使用

# [新写法] 适用于 TensorFlow >= 2.0
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMSIZE = 227

# 以下代码需要本地存在 data/train/ 和 data/test/ 目录
# 取消注释后可运行

# train_generator = ImageDataGenerator(rescale=1./255).flow_from_directory(
#     'data/train/',
#     target_size=(IMSIZE, IMSIZE),
#     batch_size=200,
#     class_mode='categorical')

# validation_generator = ImageDataGenerator(rescale=1./255).flow_from_directory(
#     'data/test/',
#     target_size=(IMSIZE, IMSIZE),
#     batch_size=200,
#     class_mode='categorical')

print('数据准备代码已就绪（需要本地数据目录）')

### 步骤3：构建AlexNet模型（参考代码示例 5-7）

下面直接构建AlexNet网络结构。输入为 227x227x3 的 RGB 图像，输出为2类分类（可根据任务修改）。

**各层详细解析：**

| 层 | 说明 | 参数量计算 |
|----|------|-----------|
| **C1**: Conv2D(96, 11x11, stride=4) | 使用大卷积核提取低级特征（边缘、纹理），步长4快速降维 | (11x11x3+1)x96 = **34,944** |
| **S1**: MaxPool(3x3, stride=2) | 最大池化下采样，保留显著特征，提供平移不变性 | 0（无参数） |
| **C2**: Conv2D(256, 5x5, same) | 中等卷积核提取中级特征（角点、轮廓组合） | (5x5x96+1)x256 = **614,656** |
| **S2**: MaxPool(3x3, stride=2) | 继续下采样 | 0 |
| **C3**: Conv2D(384, 3x3, same) | 三层连续卷积开始：不加池化，保留空间分辨率以提取更丰富的特征 | (3x3x256+1)x384 = **885,120** |
| **C4**: Conv2D(384, 3x3, same) | 连续卷积层加深网络，学习更复杂的特征组合 | (3x3x384+1)x384 = **1,327,488** |
| **C5**: Conv2D(256, 3x3, same) | 连续卷积层结束，输出高级语义特征 | (3x3x384+1)x256 = **884,992** |
| **S5**: MaxPool(3x3, stride=2) | 最后一次池化，输出6x6x256 | 0 |
| **Flatten** | 将6x6x256=9216维特征图展平为一维向量 | 0 |
| **F6**: Dense(4096) + Dropout(0.5) | 全连接层，学习全局特征组合；Dropout防止过拟合 | (9216+1)x4096 = **37,752,832** |
| **F7**: Dense(4096) + Dropout(0.5) | 第二个全连接层，进一步学习高级表示 | (4096+1)x4096 = **16,781,312** |
| **输出**: Dense(2) + Softmax | 输出各类别概率（此处2分类） | (4096+1)x2 = **8,194** |

**设计要点：** C3-C5三个卷积层连续堆叠不加池化，这是为了在不损失空间分辨率的前提下提取更深层的特征。全连接层（F6、F7）占据了约88%的参数量，是过拟合的"重灾区"，因此在这两层后都使用了Dropout。

In [ ]:
# ====== 代码示例 5-7：构建 AlexNet ======

# [旧写法] from keras.layers import Activation, Conv2D, BatchNormalization, Dense
# [旧写法] from keras.layers import Dropout, Flatten, Input, MaxPooling2D, ZeroPadding2D
# [旧写法] from keras import Model
# ↑ 独立 keras 包在 TF 2.0+ 中已不推荐使用

# [新写法] 适用于 TensorFlow >= 2.0
from tensorflow.keras.layers import Activation, Conv2D, BatchNormalization, Dense
from tensorflow.keras.layers import Dropout, Flatten, Input, MaxPooling2D, ZeroPadding2D
from tensorflow.keras import Model

IMSIZE = 227
input_layer = Input([IMSIZE, IMSIZE, 3])
x = input_layer

# 第1层：96个11×11卷积核，步长4
x = Conv2D(96, [11, 11], strides=[4, 4], activation='relu')(x)
x = MaxPooling2D([3, 3], strides=[2, 2])(x)

# 第2层：256个5×5卷积核
x = Conv2D(256, [5, 5], padding="same", activation='relu')(x)
x = MaxPooling2D([3, 3], strides=[2, 2])(x)

# 第3-5层：连续3个卷积层
x = Conv2D(384, [3, 3], padding="same", activation='relu')(x)
x = Conv2D(384, [3, 3], padding="same", activation='relu')(x)
x = Conv2D(256, [3, 3], padding="same", activation='relu')(x)
x = MaxPooling2D([3, 3], strides=[2, 2])(x)

# 全连接层
x = Flatten()(x)
x = Dense(4096, activation='relu')(x)
x = Dropout(0.5)(x)                # Dropout防止过拟合
x = Dense(4096, activation='relu')(x)
x = Dropout(0.5)(x)

# 输出层（此处以2分类为例）
x = Dense(2, activation='softmax')(x)
output_layer = x

model = Model(input_layer, output_layer)
model.summary()

### 步骤4：编译与训练（参考代码示例 5-8）

下面展示编译和训练的代码。由于没有加载实际数据，训练部分仅做编译演示。

**编译参数详解：**

- **`loss='categorical_crossentropy'`（分类交叉熵）**：用于多分类任务的标准损失函数，要求标签为one-hot编码。公式为 $L = -\sum_{i} y_i \log(\hat{y}_i)$，其中 $y_i$ 是真实标签，$\hat{y}_i$ 是预测概率。交叉熵衡量两个概率分布的差距，预测越准确则损失越小。
- **`optimizer=Adam(learning_rate=0.001)`（Adam优化器）**：Adam结合了Momentum（动量法）和RMSProp的优点，为每个参数自适应地调整学习率。`learning_rate=0.001` 是推荐的默认值。Adam的优势在于对超参数不太敏感，在大多数任务中都能获得较好的收敛效果。
- **`metrics=['accuracy']`（准确率指标）**：在训练和验证过程中额外计算分类准确率，方便监控训练效果。注意，准确率仅作为监控指标，不参与梯度计算。
- **`model.fit()` vs `model.fit_generator()`**：在TF 2.1之前，处理数据生成器需要使用 `fit_generator()`。从TF 2.1起，`fit()` 方法统一支持NumPy数组、tf.data.Dataset和数据生成器等多种输入格式。

In [ ]:
# ====== 代码示例 5-8：编译与训练 ======

# [旧写法] from keras.optimizers import Adam
# [新写法] 适用于 TensorFlow >= 2.0
from tensorflow.keras.optimizers import Adam

# [旧写法] model.compile(loss='categorical_crossentropy', optimizer=Adam(lr=0.001), metrics=['accuracy'])
# ↑ 参数 lr 在 TF 2.11+ 中已废弃
# [新写法] 适用于 TensorFlow >= 2.11
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

print('模型编译完成！')
print(f'优化器: Adam (learning_rate=0.001)')
print(f'损失函数: categorical_crossentropy')

# [旧写法] model.fit_generator(train_generator, epochs=5, validation_data=validation_generator)
# ↑ fit_generator 在 TF 2.1 中已废弃，TF 2.20 中已移除
# [新写法] 适用于 TensorFlow >= 2.1
# model.fit(train_generator, epochs=5, validation_data=validation_generator)
# ↑ 需要加载数据后才能运行训练

### 步骤5：使用随机数据验证模型前向传播

为了验证模型结构正确，我们用随机生成的数据做一次前向传播。

**前向传播（Forward Propagation）说明：**

前向传播是指输入数据从输入层开始，逐层经过网络的每一层（卷积、池化、全连接等），最终到达输出层得到预测结果的过程。在这个过程中：

1. 输入图像（4张227x227x3）进入第一层卷积层，经过96个11x11卷积核提取特征
2. 特征图依次经过后续的卷积层和池化层，空间尺寸逐渐减小，通道数（特征维度）逐渐增加
3. 经过Flatten层将三维特征图展平为一维向量
4. 通过全连接层将局部特征组合为全局特征
5. 最终Softmax层输出每个类别的概率值（和为1）

`model.predict()` 仅执行前向传播，不进行梯度计算和参数更新（即不训练），适用于推理/预测阶段。此时Dropout层不生效，所有神经元都参与计算。

In [ ]:
# 生成一批随机图像 (batch_size=4, 227x227x3)
dummy_images = np.random.rand(4, 227, 227, 3).astype('float32')

# 前向传播
predictions = model.predict(dummy_images)
print('输入形状:', dummy_images.shape)
print('输出形状:', predictions.shape)
print('预测结果（概率分布）:')
print(predictions)
print()
print('每个样本预测的类别:', np.argmax(predictions, axis=1))

---
## 五、各层参数分析

下面逐层分析 AlexNet 每一层的输出维度和参数数量。

In [ ]:
# 逐层打印每层的名称、输出形状和参数数量
print(f'{"层名称":<30} {"输出形状":<25} {"参数数量":>10}')
print('=' * 70)
total_params = 0
for layer in model.layers:
    output_shape = str(layer.output_shape) if hasattr(layer, 'output_shape') else 'N/A'
    params = layer.count_params()
    total_params += params
    print(f'{layer.name:<30} {output_shape:<25} {params:>10,}')
print('=' * 70)
print(f'{"总参数量":<55} {total_params:>10,}')
print(f'\n约 {total_params/1e6:.1f}M 个参数')

---
## 六、（可选）使用 CIFAR-10 演示 AlexNet 风格网络

### CIFAR-10 数据集简介

CIFAR-10 是由 Alex Krizhevsky（AlexNet的作者）和 Geoffrey Hinton 收集的经典图像分类数据集，包含：
- **60,000张** 32x32x3 的彩色图像（50,000张训练集 + 10,000张测试集）
- **10个类别**：飞机、汽车、鸟、猫、鹿、狗、青蛙、马、船、卡车
- 每个类别各有6,000张图像，类别分布均匀

CIFAR-10 是 TensorFlow/Keras 的内置数据集，可以通过一行代码直接加载，非常适合教学和快速实验。

### 为什么需要调整 AlexNet 来适配 CIFAR-10？

原始AlexNet设计用于处理 **227x227** 的大尺寸图像，而 CIFAR-10 的图像只有 **32x32**，尺寸差距巨大（约50倍面积差）。如果直接使用原始AlexNet：

1. **第一层卷积核太大**：原始11x11卷积核+步长4会在32x32图像上立即丧失几乎所有空间信息（输出仅6x6），无法有效提取特征
2. **多次池化导致特征图消失**：32x32的图像经过几次步长>1的操作后，特征图尺寸会缩小到0
3. **参数量过多**：原始AlexNet约6000万参数，对于仅5万张32x32图像的CIFAR-10训练集来说，严重过大，极易过拟合

因此，我们构建一个**缩小版AlexNet**：使用3x3小卷积核替代11x11，步长改为1，减少全连接层神经元数（1024替代4096），但保留AlexNet的核心设计理念（多层卷积 + ReLU + Dropout）。

> 注意：这不是完整的 AlexNet，而是为了在小图像上演示其设计理念的简化版本。

In [ ]:
# 加载 CIFAR-10 数据集
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

(X_train, Y_train), (X_test, Y_test) = cifar10.load_data()

# 归一化到 [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# One-hot 编码
Y_train_cat = to_categorical(Y_train, 10)
Y_test_cat = to_categorical(Y_test, 10)

print(f'训练集: {X_train.shape}, 标签: {Y_train_cat.shape}')
print(f'测试集: {X_test.shape}, 标签: {Y_test_cat.shape}')

# 可视化部分样本
class_names = ['飞机', '汽车', '鸟', '猫', '鹿', '狗', '青蛙', '马', '船', '卡车']
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_train[i])
    ax.set_title(class_names[Y_train[i][0]])
    ax.axis('off')
plt.suptitle('CIFAR-10 样本图像', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 构建适配 CIFAR-10 (32x32x3) 的 AlexNet 风格网络
# 保留核心设计：多层卷积 + ReLU + Dropout

input_layer_cifar = Input([32, 32, 3])
x = input_layer_cifar

# 卷积层1（缩小卷积核以适配32x32输入）
x = Conv2D(96, [3, 3], padding='same', activation='relu')(x)
x = MaxPooling2D([2, 2])(x)

# 卷积层2
x = Conv2D(256, [3, 3], padding='same', activation='relu')(x)
x = MaxPooling2D([2, 2])(x)

# 卷积层3-5（连续卷积，保留AlexNet的设计）
x = Conv2D(384, [3, 3], padding='same', activation='relu')(x)
x = Conv2D(384, [3, 3], padding='same', activation='relu')(x)
x = Conv2D(256, [3, 3], padding='same', activation='relu')(x)
x = MaxPooling2D([2, 2])(x)

# 全连接层（缩小神经元数量）
x = Flatten()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)

# 输出层（CIFAR-10 有10个类别）
x = Dense(10, activation='softmax')(x)
output_layer_cifar = x

model_cifar = Model(input_layer_cifar, output_layer_cifar)
model_cifar.summary()

In [ ]:
# 编译并训练（仅训练3个epoch作为演示）
from tensorflow.keras.optimizers import Adam

# [旧写法] model_cifar.compile(loss='categorical_crossentropy', optimizer=Adam(lr=0.001), metrics=['accuracy'])
# [新写法] 适用于 TensorFlow >= 2.11
model_cifar.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# [旧写法] model_cifar.fit_generator(...)
# [新写法] 适用于 TensorFlow >= 2.1
history = model_cifar.fit(
    X_train, Y_train_cat,
    epochs=3,
    batch_size=128,
    validation_data=(X_test, Y_test_cat)
)

In [ ]:
# 可视化训练过程
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 准确率曲线
ax1.plot(history.history['accuracy'], label='训练准确率')
ax1.plot(history.history['val_accuracy'], label='验证准确率')
ax1.set_title('准确率')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

# 损失曲线
ax2.plot(history.history['loss'], label='训练损失')
ax2.plot(history.history['val_loss'], label='验证损失')
ax2.set_title('损失')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.suptitle('AlexNet风格网络在CIFAR-10上的训练过程', fontsize=14)
plt.tight_layout()
plt.show()

---
## 七、LeNet-5 vs AlexNet 对比

| 特征 | LeNet-5 (1998) | AlexNet (2012) |
|------|----------------|----------------|
| 输入大小 | 28x28x1 | 227x227x3 |
| 深度 | 5层 | 8层 |
| 激活函数 | Sigmoid/Tanh | ReLU |
| 正则化 | 无 | Dropout |
| 参数量 | ~6万 | ~6000万 |
| 训练方式 | CPU | GPU |

---
## 八、新旧写法对照表

| 功能 | 旧写法 | 新写法 |
|------|--------|--------|
| 导入层 | `from keras.layers import Conv2D, Dense, Dropout` | `from tensorflow.keras.layers import Conv2D, Dense, Dropout` |
| 导入模型 | `from keras import Model` | `from tensorflow.keras import Model` |
| 学习率 | `Adam(lr=0.001)` | `Adam(learning_rate=0.001)` |
| 训练 | `model.fit_generator(generator, ...)` | `model.fit(generator, ...)` |
| 图像生成器 | `from keras.preprocessing.image import ImageDataGenerator` | `from tensorflow.keras.preprocessing.image import ImageDataGenerator` |
| One-hot编码 | `from keras.utils import np_utils; np_utils.to_categorical(Y)` | `from tensorflow.keras.utils import to_categorical; to_categorical(Y)` |

---
## 九、思考题

1. AlexNet相比LeNet-5有哪些关键改进？这些改进为什么重要？
2. Dropout为什么能防止过拟合？训练和测试时的行为有什么不同？
3. AlexNet为什么使用11x11的大卷积核？后续网络（如VGG）为什么不再使用大卷积核？
4. 如果训练数据量很小，AlexNet可能会出现什么问题？如何解决？